# Latent Mesh — Kaggle Training

Auto-converts HF data to binary, trains Phase 1 (250M), pushes checkpoints to HF Hub.

**Set Kaggle Secrets:**
- `HF_TOKEN` — HuggingFace write token (Settings → Secrets)
- `HF_REPO_ID` — target repo (`mohamed99raad/Latent-Mesh-Model`)

In [ ]:
# ═══════════════════════════════════════════════
# Setup
# ═══════════════════════════════════════════════
import os, sys, json, subprocess, shutil, tempfile, time
from pathlib import Path

HOME = os.path.expanduser("~")
WORK = "/kaggle/working"
os.chdir(WORK)

# ── Clone repo ──
REPO_URL = "https://github.com/mohamed-raad/latent-mesh-diffusion"
if not os.path.isdir("latent-mesh-diffusion"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir("latent-mesh-diffusion")
sys.path.insert(0, "NoProp/src")
print(f"Repo: {REPO_URL}")
print(f"Python: {sys.version}")

In [ ]:
# ── Install dependencies ──
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch>=2.5", "transformers", "datasets", "numpy",
    "accelerate", "huggingface-hub", "tensorboard", "bitsandbytes"], check=True)
print("Dependencies installed")

In [ ]:
# ── HF Hub login ──
from kaggle_secrets import UserSecretsClient
secret_client = UserSecretsClient()
HF_TOKEN = secret_client.get_secret("HF_TOKEN")
HF_REPO_ID = secret_client.get_secret("HF_REPO_ID")
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"HF Hub: {HF_REPO_ID}")

# Create model repo if needed
from huggingface_hub import HfApi
api = HfApi()
try:
    api.repo_info(HF_REPO_ID, repo_type="model")
    print(f"Repo exists: {HF_REPO_ID}")
except:
    api.create_repo(HF_REPO_ID, repo_type="model", private=True, exist_ok=True)
    print(f"Created repo: {HF_REPO_ID}")

In [ ]:
# ═══════════════════════════════════════════════
# Data: convert HF datasets → binary shards
# ═══════════════════════════════════════════════

from bin_converter import convert_hf_to_bin, BinReader
from train_pipeline import PHASE_CONFIGS

cfg = PHASE_CONFIGS["phase1_250m"]
BIN_DIR = os.path.join(WORK, "bin_data")
os.makedirs(BIN_DIR, exist_ok=True)

for ds_spec in cfg.datasets:
    hf_path = ds_spec["hf_path"]
    hf_config = ds_spec.get("hf_config")
    tag = hf_path.replace("/", "_") + (f"_{hf_config}" if hf_config else "")
    bin_path = os.path.join(BIN_DIR, tag)
    if os.path.isfile(os.path.join(bin_path, "index.json")):
        print(f"✓ Already cached: {tag}")
        continue
    print(f"Converting {tag}...")
    convert_hf_to_bin(hf_path, bin_path, hf_config=hf_config,
                      max_docs=200000, shard_size=10000,
                      max_seq_len=cfg.canvas_len)
    print(f"  ✓ {tag} done")

print(f"All data ready in {BIN_DIR}")

In [ ]:
# ═══════════════════════════════════════════════
# Training
# ═══════════════════════════════════════════════

from train_pipeline import TrainingOrchestrator

LOG_DIR = os.path.join(WORK, "logs")
CKPT_DIR = os.path.join(WORK, "checkpoints")

orchestrator = TrainingOrchestrator(
    log_dir=LOG_DIR,
    checkpoint_base=CKPT_DIR,
    use_packing=False,
)

print("Starting Phase 1 training...")
t0 = time.time()

orchestrator.run_phase(cfg)

elapsed = time.time() - t0
print(f"Training complete in {elapsed/3600:.1f}h")
print(f"Checkpoints: {CKPT_DIR}")

In [ ]:
# ═══════════════════════════════════════════════
# Push checkpoints to HF Hub
# ═══════════════════════════════════════════════

from huggingface_hub import HfApi, upload_folder

api = HfApi()

# Find latest checkpoint
phases = [d for d in os.listdir(CKPT_DIR) if d.startswith("phase_")]
for phase in sorted(phases):
    phase_dir = os.path.join(CKPT_DIR, phase)
    if os.path.isdir(phase_dir):
        print(f"Uploading {phase}...")
        upload_folder(
            folder_path=phase_dir,
            repo_id=HF_REPO_ID,
            repo_type="model",
            path_in_repo=phase,
            token=HF_TOKEN,
        )
        print(f"  ✓ {phase} pushed to {HF_REPO_ID}/{phase}")

# Also upload training logs
upload_folder(
    folder_path=LOG_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
    path_in_repo="logs",
    token=HF_TOKEN,
    ignore_patterns=["*.tfevents.*"],
)

print(f"\nAll checkpoints at: https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# ═══════════════════════════════════════════════
# Resume helper: pull previous checkpoint
# ═══════════════════════════════════════════════

if os.path.isdir("/kaggle/input/latent-mesh-checkpoints"):
    print("Found previous checkpoints in Kaggle Dataset")
    prev = "/kaggle/input/latent-mesh-checkpoints"
    for item in os.listdir(prev):
        dest = os.path.join(CKPT_DIR, item)
        if not os.path.exists(dest):
            shutil.copytree(os.path.join(prev, item), dest)
            print(f"  Restored {item}")

print("Done")